In [7]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/home/sandbox/personal-repos/packages/locallib/pandas/DBQuery.py:15: UserWarning: registration of accessor <class 'locallib.pandas.DBQuery.DBAccessor'> under name 'db' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  class DBAccessor:


In [8]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *
from locallib.box import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

In [ ]:
#Get the columns of the view and get the KPI definitions
cols = Query("PRAGMA table_info('Weekly_KPI')").execute(KPIHub_Conn)
cols.db.set_query("SELECT * FROM KPI_Definition WHERE name IN (SELECT name from temp_KPI)")
kpi_col = cols.db.execute(KPIHub_Conn, source_col = 'name', temp_table_name = 'temp_KPI')
output_dict = {kpi_col['Name']: [kpi_col['Unit'], kpi_col['Description'], kpi_col['Formula']] for _, kpi_col in kpi_col.iterrows()}


AttributeError: 'sqlite3.Connection' object has no attribute 'raw_connection'

In [ ]:
#Get total KPI
def week_dates(year, week):
    """
    Returns the start and end dates (Monday to Sunday) of the given ISO week and year.
    """
    # Ensure year and week are integers (convert if they're not)
    year = int(year)
    week = int(week)
    # Use the ISO calendar to get the Monday of the week
    # ISO: Monday is 1, Sunday is 7
    from datetime import date, timedelta
    # Python 3.8+ provides fromisocalendar
    week_start = date.fromisocalendar(year, week, 1)
    week_end = week_start + timedelta(days=6)
    # Make sure week_start is not before 2026-01-01
    min_start = date(2026, 1, 1)
    if week_start < min_start:
        week_start = min_start
        week_end = week_start + timedelta(days=6)
    return week_start, week_end


In [ ]:
regions = Query("SELECT DISTINCT BoundaryRegion FROM Weekly_KPI WHERE CustomerName = 'Cadent'").execute(KPIHub_Conn)

In [ ]:
kpi_data = Query("SELECT * FROM Weekly_KPI WHERE CustomerName = 'Cadent' AND Year = 2026").execute(KPIHub_Conn)

In [ ]:

with pd.ExcelWriter("Cadent_E_KPI.xlsx") as writer:
    # Handle 'regions' DataFrame correctly: iterate over its rows, not directly over DataFrame
    for _, region_row in regions.iterrows():
        region = region_row['BoundaryRegion']
        # Fix the ambiguity with Series comparison
        if pd.isnull(region):
            export_df = kpi_data[kpi_data['BoundaryRegion'].isnull()]
            sheet_name = "KPI Global 2026 Weekly"
        else:
            export_df = kpi_data[kpi_data['BoundaryRegion'] == region]
            sheet_name = f"KPI {region} 2026 Weekly"

        # Process the WeekDates
        export_df = export_df.copy()  # Avoid SettingWithCopyWarning
        export_df['WeekDates'] = export_df['PeriodValue'].apply(lambda week: f"{week_dates(2026, int(week))[0]} to {week_dates(2026, int(week))[1]}")
        # Move "WeekDates" to the first column in export_df
        cols = list(export_df.columns)
        if "WeekDates" in cols:
            cols.insert(0, cols.pop(cols.index("WeekDates")))
            export_df = export_df[cols]

        # Process the column name
        units = []
        for col in export_df.columns:
            if col in kpi_col['Name'].values:
                units.append(kpi_col.loc[kpi_col['Name'] == col, 'Unit'].values[0])
            else:
                units.append("")

        # Write the filtered DataFrame to the first sheet
        # Write columns and units as first two rows, then export the rest of the DataFrame
        rows = export_df.values.tolist()
        full_rows = [export_df.columns.tolist(), units] + rows
        temp_df = pd.DataFrame(full_rows)

        temp_df.to_excel(writer, sheet_name=sheet_name, index=False, header=False)

        # Post-process the sheet for bold and center alignment of the first two columns
        worksheet = writer.sheets[sheet_name]
        # Create bold and center formats
        bold_center = writer.book.add_format({'bold': True, 'align': 'center'})
        center = writer.book.add_format({'align': 'center'})

        # The first two rows (headers and units): apply bold and center format to *all* columns, not just columns 0 and 1
        for row_idx in range(2):
            for col_idx in range(len(full_rows[row_idx])):
                worksheet.write(row_idx, col_idx, full_rows[row_idx][col_idx], bold_center)  # overwrite with bold+center

        # All other rows: just center the first two columns
        for i, row in enumerate(rows, start=2):
            for col in range(2):
                worksheet.write(i, col, row[col], center)
        # Add a colored line (cell border) at the bottom of all cells in the second row (units row)
        bottom_border_format = writer.book.add_format({'bottom': 1, 'bottom_color': '#000000', 'align': 'center', 'bold': True})
        for col_idx in range(len(full_rows[1])):
            worksheet.write(1, col_idx, full_rows[1][col_idx], bottom_border_format)
    

    # Write the key and its list of [Unit, Description, Formula Used] as columns in the second sheet
    desc_rows = []
    for key, value in output_dict.items():
        if isinstance(value, list) and len(value) == 3:
            # Value is a list: [Unit, Description, Formula Used]
            row = [key] + value
        else:
            # Fallback in case the dictionary isn't formatted as expected
            row = [key, "", "", ""]
        desc_rows.append(row)
    columns = ["Key", "Unit", "Description", "Formula Used"]
    desc_df = pd.DataFrame(desc_rows, columns=columns)
    desc_df.to_excel(writer, sheet_name="KPI Descriptions", index=False)

box_obj = BoxFile(local_path = "Cadent_E_KPI.xlsx", box_file_id = 2266637913401)
box_obj.upload()
